Market Data Aggregation
=======================

Purpose:
Transform raw market offer data collected from SimCompanies into
aggregated market snapshots suitable for further analytical processing.

Input:
- Raw CSV files containing market offers from the Bronze layer.
- Product mapping table linking external API resource IDs to internal
  product IDs.

Process:
1. Load all available raw market offer files.
2. Validate data types and required columns.
3. Remove duplicate offers within individual snapshots.
4. Map external API resource IDs to internal product IDs.
5. Aggregate market offers by snapshot, product, quality and offer kind.
6. Calculate market statistics:
   - Quantity-weighted average price
   - Median price (approximate)
   - Minimum and maximum price
   - Total offered quantity
   - Number of offers
7. Preserve snapshot timestamps and product identifiers.
8. Export the aggregated dataset to the Silver layer in Parquet format,
   partitioned by snapshot ID.

Output:
A curated market dataset containing aggregated offer statistics
for each product, quality level, offer kind and market snapshot.

Important:
The source data represents active market offers, not completed
transactions. Therefore, aggregated prices and quantities describe
observed market supply rather than actual trading volume or demand.

Snapshots are kept separate to preserve historical market observations.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
    .appName("SimCompanies Market Aggregation") \
    .getOrCreate()

snapshot_id	product_id	quality	kind	avg_price	median_price	min_price	max_price	total_quantity	offer_count

In [3]:
from pyspark.sql.readwriter import DataFrameReader

reader: DataFrameReader = spark.read

In [4]:
df = (
    reader
    .format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(r'..\..\data\bronze\market\offers\\')
)

In [5]:
df.head(5)

[Row(snapshot_id='20261122T184628Z', captured_at=datetime.datetime(2026, 11, 22, 19, 46, 48), source_page_id=1, api_resource_id=1, order_id=128376915, kind=1, quantity=504697, quality=0, price=0.274, posted=datetime.datetime(2026, 11, 22, 19, 43, 46, 170790), datetime_decay_updated=datetime.datetime(2026, 11, 22, 19, 43, 46, 170790), fees=5531, seller_id=5214748, seller_company='Sampling DU4x', seller_realm_id=0, seller_npc=False),
 Row(snapshot_id='20261122T184628Z', captured_at=datetime.datetime(2026, 11, 22, 19, 46, 48), source_page_id=1, api_resource_id=1, order_id=128376805, kind=1, quantity=3490, quality=1, price=0.275, posted=datetime.datetime(2026, 11, 22, 19, 40, 2, 812238), datetime_decay_updated=datetime.datetime(2026, 11, 22, 19, 40, 2, 812238), fees=37, seller_id=4403493, seller_company='mad.s', seller_realm_id=0, seller_npc=False),
 Row(snapshot_id='20261122T184628Z', captured_at=datetime.datetime(2026, 11, 22, 19, 46, 48), source_page_id=1, api_resource_id=1, order_id=12

In [6]:
df.createOrReplaceTempView("market")

In [7]:
result = spark.sql("""
    SELECT *
    FROM market
    LIMIT 10
""")

result.show()

+----------------+-------------------+--------------+---------------+---------+----+--------+-------+-----+--------------------+----------------------+-----+---------+--------------------+---------------+----------+
|     snapshot_id|        captured_at|source_page_id|api_resource_id| order_id|kind|quantity|quality|price|              posted|datetime_decay_updated| fees|seller_id|      seller_company|seller_realm_id|seller_npc|
+----------------+-------------------+--------------+---------------+---------+----+--------+-------+-----+--------------------+----------------------+-----+---------+--------------------+---------------+----------+
|20261122T184628Z|2026-11-22 19:46:48|             1|              1|128376915|   1|  504697|      0|0.274|2026-11-22 19:43:...|  2026-11-22 19:43:...| 5531|  5214748|       Sampling DU4x|              0|     false|
|20261122T184628Z|2026-11-22 19:46:48|             1|              1|128376805|   1|    3490|      1|0.275|2026-11-22 19:40:...|  2026-1

snapshot_id	product_id	quality	kind	avg_price	median_price	min_price	max_price	total_quantity	offer_count

In [8]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [9]:
result = spark.sql("""
    SELECT
        snapshot_id,
        TO_TIMESTAMP(snapshot_id, "yyyyMMdd'T'HHmmss'Z'") AS snapshot_timestamp,
        api_resource_id,
        quality,
        kind,

        SUM(quantity * price) / SUM(quantity) AS avg_price,
        PERCENTILE_APPROX(price, 0.5) AS median_price,

        MIN(price) AS min_price,
        MAX(price) AS max_price,

        SUM(quantity) AS total_quantity,
        COUNT(*) AS offer_count

    FROM market

    GROUP BY
        snapshot_id,
        api_resource_id,
        quality,
        kind
""")

result.show()

+----------------+-------------------+---------------+-------+----+-------------------+------------+---------+---------+--------------+-----------+
|     snapshot_id| snapshot_timestamp|api_resource_id|quality|kind|          avg_price|median_price|min_price|max_price|total_quantity|offer_count|
+----------------+-------------------+---------------+-------+----+-------------------+------------+---------+---------+--------------+-----------+
|20261104T130919Z|2026-11-04 13:09:19|              1|      0|   1|0.26848106549291195|       0.268|     0.26|    0.272|      59818061|         63|
|20261104T130919Z|2026-11-04 13:09:19|              1|      1|   1| 0.2665676785941328|       0.269|     0.26|    0.272|       6244914|         10|
|20261104T130919Z|2026-11-04 13:09:19|              1|      2|   1| 0.2683023331107683|       0.267|    0.261|    0.272|     125737022|         43|
|20261104T130919Z|2026-11-04 13:09:19|              1|      3|   1| 0.2697465059534517|        0.27|    0.265|  

In [10]:
df.write.partitionBy("snapshot_id").parquet(r'..\..\data\silver\market\\', mode='overwrite')